# 02 计算盗垒成功率

## 这一步做了什么
从 MLB Stats API 的比赛逐回合记录中提取全部盗垒尝试，计算 2025 年 6 月全联盟的盗垒成功率，
并按战术类型、球队、跑者汇总。

## 数据从哪来
- 比赛清单与球队信息：`data/raw/2025-06_statcast_month.csv`
- 盗垒事件：MLB Stats API 的 play-by-play 接口
  `https://statsapi.mlb.com/api/v1/game/{gamePk}/playByPlay`

## 为什么不使用 Statcast 的 events 列
本项目的 Statcast 导出中，`events` 只记录打席终结事件。全月 30031 条事件里，
`stolen_base_*`、`caught_stealing_*`、`pickoff_*` 全部为 0，跑垒事件没有任何标签。
MLB play-by-play 把每次跑垒记录为独立的 play，`result.event` 直接给出事件名，因此改用它。

## 结果是什么
- `data/processed/2025-06_sb_summary.csv`：总体成功率与 Wilson 区间
- `data/processed/2025-06_sb_by_team.csv`：分球队
- `data/processed/2025-06_sb_by_runner.csv`：分跑者（3 次及以上尝试）
- `data/processed/2025-06_sb_attempts.csv`：逐次尝试明细

## 统计口径
成功率 = 盗垒成功 / 盗垒尝试。
分母包含盗垒成功、盗垒失败（Caught Stealing）、牵制致盗垒失败（Pickoff Caught Stealing）。
不含纯牵制出局（Pickoff 1B/2B/3B），因为跑者没有起跑。


In [2]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import requests


def find_project_root(marker="requirements.txt") -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    return p


ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PBP_DIR = RAW_DIR / "mlb_pbp"
for d in (PROC_DIR, PBP_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATA_LABEL = "2025-06"

print("项目根目录:", ROOT)


项目根目录: C:\Users\GOOD\PycharmProjects\Baseball-Analytics


In [3]:
df = pd.read_csv(RAW_DIR / f"{DATA_LABEL}_statcast_month.csv", low_memory=False)
df["game_date"] = pd.to_datetime(df["game_date"])
df = df.drop_duplicates()

n_missing_outs = int(df["outs_when_up"].isna().sum())
df["outs_when_up"] = df["outs_when_up"].fillna(0).astype(int)

for b in ["on_1b", "on_2b", "on_3b"]:
    df[f"has_{b}"] = df[b].notna().astype(int)

games = (df.groupby("game_pk")
           .agg(game_date=("game_date", "first"),
                home_team=("home_team", "first"),
                away_team=("away_team", "first"),
                pitches=("pitch_number", "size"))
           .reset_index())

print("逐球记录:", df.shape)
print("比赛场数:", len(games))
print("outs_when_up 缺失:", n_missing_outs, "行")
print("日期范围:", games["game_date"].min().date(), "~", games["game_date"].max().date())


逐球记录: (115816, 31)
比赛场数: 397
outs_when_up 缺失: 0 行
日期范围: 2025-06-01 ~ 2025-06-30


In [4]:
def fetch_pbp(game_pk: int, retries: int = 3):
    """下载一场比赛的 play-by-play。成功后在本地缓存，重复运行不再发请求。"""
    fp = PBP_DIR / f"{game_pk}.json"
    if fp.exists():
        return json.loads(fp.read_text(encoding="utf-8"))

    url = f"https://statsapi.mlb.com/api/v1/game/{game_pk}/playByPlay"
    for i in range(retries):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                fp.write_text(r.text, encoding="utf-8")
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(1 + i)
    return None


game_pks = games["game_pk"].astype(int).tolist()
failed = []

for i, gpk in enumerate(game_pks, 1):
    if fetch_pbp(gpk) is None:
        failed.append(gpk)
    if i % 50 == 0:
        print(f"{i}/{len(game_pks)}")

print("抓取完成 | 失败:", len(failed), "场")
if failed:
    print("失败场次:", failed[:20])


50/397
100/397
150/397
200/397
250/397
300/397
350/397
抓取完成 | 失败: 2 场
失败场次: [777503, 777504]


In [5]:
records = []

for gpk in game_pks:
    j = fetch_pbp(gpk)
    if j is None:
        continue

    for play in j.get("allPlays", []):
        ev = (play.get("result") or {}).get("event", "")
        about = play.get("about") or {}
        count = play.get("count") or {}

        runner_id = runner_name = origin = None
        for r in play.get("runners") or []:
            details = r.get("details") or {}
            if details.get("event") == ev:
                info = details.get("runner") or {}
                runner_id = info.get("id")
                runner_name = info.get("fullName")
                origin = (r.get("movement") or {}).get("originBase")
                break

        records.append({
            "game_pk": gpk,
            "event": ev,
            "inning": about.get("inning"),
            "half": about.get("halfInning"),
            "outs_before": count.get("outs"),
            "origin_base": origin,
            "runner_id": runner_id,
            "runner_name": runner_name,
        })

pbp = pd.DataFrame(records)
print("play 总数:", len(pbp))

running = pbp[pbp["event"].str.contains("Stolen|Caught Stealing|Pickoff", na=False)]
print("\n跑垒相关事件分布:")
print(running["event"].value_counts().to_string())


play 总数: 30051

跑垒相关事件分布:
event
Caught Stealing 2B            35
Pickoff 1B                    14
Pickoff Caught Stealing 2B     8
Pickoff 2B                     2
Caught Stealing 3B             2
Pickoff Caught Stealing 3B     2
Caught Stealing Home           2
Stolen Base 2B                 1


In [6]:
EVENT_TO_BASE = {
    "Stolen Base 2B": 2, "Stolen Base 3B": 3, "Stolen Base Home": 4,
    "Caught Stealing 2B": 2, "Caught Stealing 3B": 3, "Caught Stealing Home": 4,
    "Pickoff Caught Stealing 2B": 2, "Pickoff Caught Stealing 3B": 3,
    "Pickoff Caught Stealing Home": 4,
}
SUCCESS_EVENTS = {"Stolen Base 2B", "Stolen Base 3B", "Stolen Base Home"}

att = pbp[pbp["event"].isin(EVENT_TO_BASE)].copy()

if len(att) == 0:
    raise ValueError(
        "没有匹配到任何盗垒事件。查看上一格打印的事件名，补进 EVENT_TO_BASE 再重跑。"
    )

att["target_base"] = att["event"].map(EVENT_TO_BASE)
att["success"] = att["event"].isin(SUCCESS_EVENTS).astype(int)
att["inning_topbot"] = att["half"].map({"top": "Top", "bottom": "Bot"})
att["outs_before"] = att["outs_before"].fillna(0).astype(int)

att = att.merge(games[["game_pk", "game_date", "home_team", "away_team"]],
                on="game_pk", how="left")
att["batting_team"] = np.where(att["inning_topbot"].eq("Top"),
                               att["away_team"], att["home_team"])

att = (att.sort_values(["game_date", "game_pk", "inning", "target_base"])
          .reset_index(drop=True))

print("盗垒尝试:", len(att))
print("成功:", int(att["success"].sum()), "| 失败:", int((1 - att["success"]).sum()))
print("球队缺失:", int(att["batting_team"].isna().sum()),
      "| 跑者缺失:", int(att["runner_id"].isna().sum()))
print("\n失败事件的出局数分布（若出现 3，说明该字段是动作之后的值）:")
print(att.loc[att["success"] == 0, "outs_before"].value_counts().sort_index().to_string())


盗垒尝试: 50
成功: 1 | 失败: 49
球队缺失: 0 | 跑者缺失: 0

失败事件的出局数分布（若出现 3，说明该字段是动作之后的值）:
outs_before
3    49


In [7]:
EVENT_TO_BASE = {
    "Stolen Base 2B": 2, "Stolen Base 3B": 3, "Stolen Base Home": 4,
    "Caught Stealing 2B": 2, "Caught Stealing 3B": 3, "Caught Stealing Home": 4,
    "Pickoff Caught Stealing 2B": 2, "Pickoff Caught Stealing 3B": 3,
    "Pickoff Caught Stealing Home": 4,
}
SUCCESS_EVENTS = {"Stolen Base 2B", "Stolen Base 3B", "Stolen Base Home"}

att = pbp[pbp["event"].isin(EVENT_TO_BASE)].copy()

if len(att) == 0:
    raise ValueError(
        "没有匹配到任何盗垒事件。查看上一格打印的事件名，补进 EVENT_TO_BASE 再重跑。"
    )

att["target_base"] = att["event"].map(EVENT_TO_BASE)
att["success"] = att["event"].isin(SUCCESS_EVENTS).astype(int)
att["inning_topbot"] = att["half"].map({"top": "Top", "bottom": "Bot"})
att["outs_before"] = att["outs_before"].fillna(0).astype(int)

att = att.merge(games[["game_pk", "game_date", "home_team", "away_team"]],
                on="game_pk", how="left")
att["batting_team"] = np.where(att["inning_topbot"].eq("Top"),
                               att["away_team"], att["home_team"])

att = (att.sort_values(["game_date", "game_pk", "inning", "target_base"])
          .reset_index(drop=True))

print("盗垒尝试:", len(att))
print("成功:", int(att["success"].sum()), "| 失败:", int((1 - att["success"]).sum()))
print("球队缺失:", int(att["batting_team"].isna().sum()),
      "| 跑者缺失:", int(att["runner_id"].isna().sum()))
print("\n失败事件的出局数分布（若出现 3，说明该字段是动作之后的值）:")
print(att.loc[att["success"] == 0, "outs_before"].value_counts().sort_index().to_string())


盗垒尝试: 50
成功: 1 | 失败: 49
球队缺失: 0 | 跑者缺失: 0

失败事件的出局数分布（若出现 3，说明该字段是动作之后的值）:
outs_before
3    49


In [8]:
# 用 Statcast 侧的垒位变化数做量级对照
KEY = ["game_pk", "inning", "inning_topbot"]
d = df.sort_values(KEY + ["at_bat_number", "pitch_number"]).reset_index(drop=True)
g = d.groupby(KEY, sort=False)

nxt = g[["on_1b", "on_2b", "on_3b"]].shift(-1)
moved = (d[["on_1b", "on_2b", "on_3b"]].isna().to_numpy()
         != nxt.isna().to_numpy()).any(axis=1)
same_pa = g["at_bat_number"].shift(-1).eq(d["at_bat_number"]).to_numpy()

mid_pa_moves = int((moved & same_pa & d["events"].isna().to_numpy()).sum())

print(f"Statcast 中含垒位变化的打席内投球: {mid_pa_moves}")
print(f"MLB API 中识别的盗垒尝试: {len(att)}")
print("前者还包括暴投、野传球、纯牵制，所以应当大于后者")


Statcast 中含垒位变化的打席内投球: 886
MLB API 中识别的盗垒尝试: 50
前者还包括暴投、野传球、纯牵制，所以应当大于后者


In [9]:
def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


n_att = len(att)
n_sb = int(att["success"].sum())
rate = n_sb / n_att
lo, hi = wilson_ci(n_sb, n_att)

print(f"{DATA_LABEL} 盗垒成功率 = {n_sb}/{n_att} = {rate:.1%}")
print(f"95% Wilson 区间: [{lo:.1%}, {hi:.1%}]")
print("近年大联盟整体参考值: 78% ~ 81%")

by_type = (att.groupby("target_base")
              .agg(attempts=("success", "size"), SB=("success", "sum")))
by_type["CS"] = by_type["attempts"] - by_type["SB"]
by_type["rate"] = (by_type["SB"] / by_type["attempts"]).round(4)
by_type.index = ["盗二垒", "盗三垒", "盗本垒"]

print("\n按战术类型:")
print(by_type.to_string())


2025-06 盗垒成功率 = 1/50 = 2.0%
95% Wilson 区间: [0.4%, 10.5%]
近年大联盟整体参考值: 78% ~ 81%

按战术类型:
     attempts  SB  CS    rate
盗二垒        44   1  43  0.0227
盗三垒         4   0   4  0.0000
盗本垒         2   0   2  0.0000


In [10]:
team_stats = (att.groupby("batting_team")
                 .agg(attempts=("success", "size"), SB=("success", "sum")))
team_stats["CS"] = team_stats["attempts"] - team_stats["SB"]
team_stats["rate"] = (team_stats["SB"] / team_stats["attempts"]).round(4)
team_stats = team_stats.sort_values("attempts", ascending=False)

print(f"共 {len(team_stats)} 支球队有盗垒尝试，前 15：")
print(team_stats.head(15).to_string())


共 23 支球队有盗垒尝试，前 15：
              attempts  SB  CS  rate
batting_team                        
WSH                  5   0   5   0.0
LAA                  4   0   4   0.0
AZ                   4   0   4   0.0
NYY                  4   0   4   0.0
MIL                  3   0   3   0.0
CWS                  3   0   3   0.0
MIA                  3   0   3   0.0
TOR                  2   0   2   0.0
STL                  2   0   2   0.0
BOS                  2   0   2   0.0
CLE                  2   0   2   0.0
ATH                  2   0   2   0.0
PHI                  2   0   2   0.0
SD                   2   0   2   0.0
TB                   2   0   2   0.0


In [11]:
runner_stats = (att.dropna(subset=["runner_id"])
                   .groupby(["runner_id", "runner_name"])
                   .agg(attempts=("success", "size"), SB=("success", "sum"))
                   .reset_index())
runner_stats["CS"] = runner_stats["attempts"] - runner_stats["SB"]
runner_stats["rate"] = (runner_stats["SB"] / runner_stats["attempts"]).round(4)
runner_stats = (runner_stats[runner_stats["attempts"] >= 3]
                .sort_values("attempts", ascending=False))

print(f"尝试 3 次及以上的跑者共 {len(runner_stats)} 人，前 15：")
print(runner_stats.head(15).to_string(index=False))


尝试 3 次及以上的跑者共 2 人，前 15：
 runner_id runner_name  attempts  SB  CS  rate
  687263.0   Zach Neto         4   0   4   0.0
  695734.0 Daylen Lile         3   0   3   0.0


In [12]:
summary = pd.DataFrame([{
    "period": DATA_LABEL,
    "games": len(games),
    "attempts": n_att,
    "SB": n_sb,
    "CS": n_att - n_sb,
    "rate": round(rate, 4),
    "ci_low": round(lo, 4),
    "ci_high": round(hi, 4),
}])

summary.to_csv(PROC_DIR / f"{DATA_LABEL}_sb_summary.csv", index=False, encoding="utf-8-sig")
team_stats.to_csv(PROC_DIR / f"{DATA_LABEL}_sb_by_team.csv", encoding="utf-8-sig")
runner_stats.to_csv(PROC_DIR / f"{DATA_LABEL}_sb_by_runner.csv", index=False, encoding="utf-8-sig")
att.to_csv(PROC_DIR / f"{DATA_LABEL}_sb_attempts.csv", index=False, encoding="utf-8-sig")

print("已保存:")
for n in ["sb_summary", "sb_by_team", "sb_by_runner", "sb_attempts"]:
    print(f"  data/processed/{DATA_LABEL}_{n}.csv")

print("\n=== 结论 ===")
print(f"{DATA_LABEL} 共 {len(games)} 场比赛，盗垒尝试 {n_att} 次。")
print(f"成功率 {rate:.1%}，95% 区间 [{lo:.1%}, {hi:.1%}]。")


已保存:
  data/processed/2025-06_sb_summary.csv
  data/processed/2025-06_sb_by_team.csv
  data/processed/2025-06_sb_by_runner.csv
  data/processed/2025-06_sb_attempts.csv

=== 结论 ===
2025-06 共 397 场比赛，盗垒尝试 50 次。
成功率 2.0%，95% 区间 [0.4%, 10.5%]。
